In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Summarization Middleware

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

In [3]:
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model = "llama-3.3-70b-versatile",
    model_provider = "groq"
)                           
agent = create_agent(
    model = model,
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger = ("messages", 10),
            keep = ("messages", 4)
        )
    ]
    
)

In [4]:
config = {"configurable": {"thread_id": "test_1"}}

In [5]:
questions = [
    "what is 2+2",
    "what is 10/2",
    "what is 100/4",
    "who is dean jones",
    "what is 100+21*0+1"    
]

for i in questions:
    response = agent.invoke({"messages": [HumanMessage(content = i)]}, config)
    print(response["messages"][-1].content)

2 + 2 = 4
10 / 2 = 5
100 / 4 = 25
Dean Jones (1931-2015) was an American actor, best known for his roles in several Disney films, particularly as the star of the "Herbie" series. Some of his notable films include:

1. The Love Bug (1969) - He played the role of Jim Douglas, the owner of the Volkswagen Beetle named Herbie.
2. The $1,000,000 Duck (1971)
3. Snowball Express (1972)
4. The Shaggy D.A. (1976)

He also appeared in other films and television shows, but his work with Disney remains his most iconic and enduring legacy.

However, there is also another notable person with the name Dean Jones, an Australian cricketer, who played for the Australian national team from 1984 to 1994. He was a talented batsman and is still involved in the cricket world as a commentator and analyst.
To calculate this, we need to follow the order of operations (PEMDAS):

1. **Multiply 21 and 0**: 21**0 = 0
2. **Add 100 and 0**: 100 + 0 = 100
3. **Add 1 to 100**: 100 + 1 = 101

So, 100 + 21**0 + 1 = 101


In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model

In [7]:
model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider = "groq"
)

@tool
def search_hotel(city: str) -> str:
    """Search hotels: returns long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, 350$ per night, spa, pool, gym.
    2. City Inn - 4 star , 180$ per night, business center.
    3. Budget Stay- 3 star, 75$ per night, free wifi"""

agent = create_agent(
    model = model,
    tools = [search_hotel],
    checkpointer= InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = model,
            trigger = ("tokens", 1000),
            keep = ("tokens", 200)
        )
    ]

)

config = {"configurable" : {"thread_id": "test_3"}}

def token_count(messages):
    token_char = 0
    for message in messages:
        token_char += len(str(message.content))
    return token_char // 4

In [8]:
cities = ["Paris", "London"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"find hotels in {city}") ]},
        config=config
    )

    tokens = token_count(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(response["messages"])

Paris: ~159 tokens, 6 messages
[HumanMessage(content='find hotels in Paris', additional_kwargs={}, response_metadata={}, id='a628527c-ea2a-4ec7-a5bb-85c93a006ea6'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'a6vxb3a60', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotel'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 226, 'total_tokens': 241, 'completion_time': 0.035008028, 'completion_tokens_details': None, 'prompt_time': 0.011559839, 'prompt_tokens_details': None, 'queue_time': 0.397969674, 'total_time': 0.046567867}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fdd18-e466-7b02-9212-85b141705626-0', tool_calls=[{'name': 'search_hotel', 'args': {'city': 'Paris'}, 'id': 'a6vxb3a60', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model

In [14]:
model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider= "groq"
)

@tool
def search_hotel(city: str) -> str:
    """Search Hotels"""
    return f"""Hotel in {city}:
    1. Grand Hotel: 500$ per night, swimming pool, AC.
    2. Medium Hotel: 200$ per night, free wifi.
    3. Cheap Hotel: 5$ per night, free security."""

agent = create_agent(
    model = model,
    tools= [search_hotel],
    checkpointer = InMemorySaver(),
    middleware= [
        SummarizationMiddleware(
            model = model,
            trigger = ("fraction", 0.005),
            keep = ("fraction", 0.002)
        )
    ]

)

config = {"configurable": {"thread_id" : "test_4"}}

def token_count(messages):
    token_char = 0
    for message in messages:
        token_char += len(str(message.content)) 
    return token_char // 4     


In [16]:
cities = ["Paris", "London","Karachi","Moscow"]
for city in cities:
    response = agent.invoke(
        {"messages" : [HumanMessage(content = f"Find the hotel in {city}")]},
        config = config
    )
    tokens = token_count(response["messages"])
    print(f"{city} ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{city} ~{tokens} tokens, {len(response['messages'])}")


Paris ~208 tokens, 6 messages
Paris ~208 tokens, 6
London ~375 tokens, 10 messages
London ~375 tokens, 10
Karachi ~489 tokens, 9 messages
Karachi ~489 tokens, 9
Moscow ~518 tokens, 9 messages
Moscow ~518 tokens, 9
